# Retrieval-Quality Judge (AI-judge comparison)

**Part of:** RAG chunking-strategy evaluation for the "AI Engineering" book chatbot project.

**What this notebook does:**
1. Loads the question bank generated by `01_Data_Preparation.ipynb`
2. Samples a random subset of questions
3. For each question, retrieves top-k chunks from all three Chroma collections
   (fixed-size, 3-sentence, single-sentence)
4. Asks a local LLM (via Ollama) to judge which set of retrieved chunks best
   supports answering the question — with method labels randomized per question
   to avoid positional bias
5. Aggregates win counts across the sample

**Depends on:** `chroma_db` (built by the three chunking notebooks) and
`questions_checkpoint.jsonl` (built by `01_Data_Preparation.ipynb`) both already
existing on disk.

## Imports

In [1]:
import json
import random
from pathlib import Path
from collections import Counter

import chromadb
import ollama
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

D:\pytorch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load and sample questions

Loads the full question bank, then takes a random sample. Fixed seed so the
sample is reproducible across reruns.

In [2]:
QUESTIONS_PATH = Path("questions_checkpoint.jsonl")

def load_questions(path: Path) -> list[dict]:
    all_questions = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            for q in row["questions"]:
                all_questions.append({"page": row["page"], **q})
    return all_questions

all_questions = load_questions(QUESTIONS_PATH)
print(f"Loaded {len(all_questions)} total questions")

sample_questions = all_questions  # judging the full bank now, not a sample
print(f"Judging {len(sample_questions)} questions")

Loaded 1145 total questions
Judging 1145 questions


## Connect to the three Chroma collections

Loads the same embedding model used to build the collections, so query
embeddings live in the same space.

In [3]:
CHROMA_PATH = "./Chunking/chroma_db"  # verify this path exists — see note above

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

collections = {
    "fixed_size": chroma_client.get_collection("fixed_size_first5"),
    "three_sentence": chroma_client.get_collection("three_sentence_first5"),
    "single_sentence": chroma_client.get_collection("single_sentence_first5"),
}

for name, coll in collections.items():
    print(f"{name}: {coll.count()} chunks")

fixed_size: 1213 chunks
three_sentence: 1726 chunks
single_sentence: 4918 chunks


## Retrieval helper

Queries all three collections with the same embedded question and returns
their top-k retrieved passages.

In [4]:
def retrieve(question: str, top_k: int = 5) -> dict[str, list[str]]:
    query_embedding = embedding_model.encode([question]).tolist()
    results = {}
    for method, coll in collections.items():
        r = coll.query(query_embeddings=query_embedding, n_results=top_k)
        results[method] = r["documents"][0]
    return results

## Judge setup

Randomizes which method gets labeled A/B/C per question (avoids positional
bias), truncates each passage to 400 chars so the prompt stays a reasonable
size, and asks for strict JSON so the verdict is parseable.

In [5]:
JUDGE_MODEL = "llama3"

EVALUATION_GUIDELINE = """
You are evaluating RETRIEVAL quality for a RAG system — NOT answer quality.
You are not being shown a generated answer. You are being shown raw retrieved
passages, and judging whether they would let someone WRITE a good answer.

Score each set of passages on three criteria, 1-5 each:

1. RELEVANCE — Do the passages relate to the question's topic at all?
   5 = Every passage is clearly on-topic for the question.
   3 = Most passages are on-topic, but one or two drift to unrelated material.
   1 = Passages are mostly about a different topic than the question.

2. COVERAGE — Do the passages contain the actual fact/explanation needed to
   answer the question, not just something adjacent to it?
   5 = The specific information needed to fully answer the question is present.
   3 = Passages are in the right neighborhood but only partially answer it,
       or require outside knowledge to connect the dots.
   1 = Nothing in the passages helps answer the question, even indirectly.

3. PRECISION — How much of the retrieved text is useful signal vs. noise
   (cut-off sentences, headers, table-of-contents fragments, boilerplate)?
   5 = Nearly all retrieved text is substantive and readable on its own.
   3 = A mix — some passages are clean, others are fragments or noise.
   1 = Mostly noise — fragments, headers, or text that doesn't stand alone.

Do NOT judge writing style, length, or which set "sounds" more complete —
score strictly against the three definitions above.
"""

JUDGE_PROMPT = """{guideline}

Question: {question}

Set A:
{set_a}

Set B:
{set_b}

Set C:
{set_c}

For EACH set, give scores for relevance, coverage, and precision (1-5 each),
plus a one-sentence note justifying the coverage score specifically (the
hardest criterion to get right). Respond with strict JSON, no other text:

{{
  "A": {{"relevance": <1-5>, "coverage": <1-5>, "precision": <1-5>, "note": "..."}},
  "B": {{"relevance": <1-5>, "coverage": <1-5>, "precision": <1-5>, "note": "..."}},
  "C": {{"relevance": <1-5>, "coverage": <1-5>, "precision": <1-5>, "note": "..."}}
}}
"""

def format_set(chunks: list[str]) -> str:
    return "\n\n".join(f"[{i+1}] {c[:400]}" for i, c in enumerate(chunks))

def build_judge_prompt(question: str, retrieved: dict[str, list[str]]):
    methods = list(retrieved.keys())
    random.shuffle(methods)
    label_map = dict(zip(["A", "B", "C"], methods))

    prompt = JUDGE_PROMPT.format(
        guideline=EVALUATION_GUIDELINE,
        question=question,
        set_a=format_set(retrieved[label_map["A"]]),
        set_b=format_set(retrieved[label_map["B"]]),
        set_c=format_set(retrieved[label_map["C"]]),
    )
    return prompt, label_map

import re

def judge(prompt: str) -> dict:
    response = ollama.chat(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        format="json",                                      # force valid JSON output
        options={"temperature": 0.0, "num_predict": 1024},   # default num_predict was truncating the response
    )
    raw = response["message"]["content"].strip()

    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:].strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # fallback: pull the outermost {...} block in case anything stray slipped in
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    return {"parse_error": raw}

### Sanity-check on one question before running the full sample

In [6]:
test_q = sample_questions[0]
test_retrieved = retrieve(test_q["question"])
test_prompt, test_label_map = build_judge_prompt(test_q["question"], test_retrieved)

print(test_prompt)
print("\n--- label map ---")
print(test_label_map)

test_verdict = judge(test_prompt)
print("\n--- verdict ---")
print(test_verdict)


You are evaluating RETRIEVAL quality for a RAG system — NOT answer quality.
You are not being shown a generated answer. You are being shown raw retrieved
passages, and judging whether they would let someone WRITE a good answer.

Score each set of passages on three criteria, 1-5 each:

1. RELEVANCE — Do the passages relate to the question's topic at all?
   5 = Every passage is clearly on-topic for the question.
   3 = Most passages are on-topic, but one or two drift to unrelated material.
   1 = Passages are mostly about a different topic than the question.

2. COVERAGE — Do the passages contain the actual fact/explanation needed to
   answer the question, not just something adjacent to it?
   5 = The specific information needed to fully answer the question is present.
   3 = Passages are in the right neighborhood but only partially answer it,
       or require outside knowledge to connect the dots.
   1 = Nothing in the passages helps answer the question, even indirectly.

3. PRECISI

## Run the judge over the sample

In [7]:
def total_score(scores: dict) -> int:
    return scores.get("relevance", 0) + scores.get("coverage", 0) + scores.get("precision", 0)

RESULTS_PATH = Path("judge_results_checkpoint.jsonl")
CHECKPOINT_EVERY = 20

# resume support: reload anything already judged, skip it this run
judge_results = []
already_done = set()
if RESULTS_PATH.exists():
    with open(RESULTS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            judge_results.append(row)
            already_done.add(row["question"])
    print(f"Resuming: {len(already_done)} questions already judged")

remaining = [q for q in sample_questions if q["question"] not in already_done]

with open(RESULTS_PATH, "a", encoding="utf-8") as f:
    for i, q in enumerate(tqdm(remaining, desc="Judging retrieval"), start=1):
        retrieved = retrieve(q["question"])
        prompt, label_map = build_judge_prompt(q["question"], retrieved)
        verdict = judge(prompt)

        if "parse_error" in verdict:
            result = {
                "question": q["question"], "page": q["page"],
                "label_map": label_map, "verdict": verdict, "parse_error": True,
            }
        else:
            scores_by_method = {label_map[label]: scores for label, scores in verdict.items()}
            winner_method = max(scores_by_method, key=lambda m: total_score(scores_by_method[m]))
            result = {
                "question": q["question"],
                "page": q["page"],
                "label_map": label_map,
                "scores_by_method": scores_by_method,
                "winner_method": winner_method,
                "parse_error": False,
            }

        judge_results.append(result)
        f.write(json.dumps(result) + "\n")

        if i % CHECKPOINT_EVERY == 0:
            f.flush()
            print(f"Checkpoint: {i}/{len(remaining)} this run ({len(judge_results)} total)")

print("Done.")

Judging retrieval: 100%|███████████████████████████████████████████████████████████████| 30/30 [27:45<00:00, 55.53s/it]

Done.


## Aggregate results

In [9]:
from collections import defaultdict

parse_errors = [r for r in judge_results if r.get("parse_error")]
valid_results = [r for r in judge_results if not r.get("parse_error")]
print(f"{len(parse_errors)} parse errors out of {len(judge_results)}")

wins = Counter(r["winner_method"] for r in valid_results)
print("\nWin counts:", dict(wins))

criterion_sums = defaultdict(lambda: defaultdict(list))
for r in valid_results:
    for method, scores in r["scores_by_method"].items():
        for criterion in ("relevance", "coverage", "precision"):
            criterion_sums[method][criterion].append(scores.get(criterion, 0))

print("\nAverage scores per method:")
for method, criteria in criterion_sums.items():
    avgs = {c: round(sum(v) / len(v), 2) for c, v in criteria.items()}
    print(f"  {method}: {avgs}")

30 parse errors out of 30

Win counts: {}

Average scores per method:
